In [4]:
!pip install xgboost pandas numpy scikit-learn

In [6]:
from google.colab import files

uploaded = files.upload()

Saving placement_predict_50k_adjusted.csv to placement_predict_50k_adjusted (2).csv


In [9]:
import time
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from xgboost import XGBClassifier


# Constants
RANDOM_STATE = 42
TARGET_COL = "PlacementStatus"
DATA_PATH = "placement_predict_50k_adjusted (2).csv"


# Load dataset
df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)


# Remove IsAnomaly column if present
if "IsAnomaly" in df.columns:
    df = df.drop(columns=["IsAnomaly"])


# Separate target and features
y = df[TARGET_COL].astype(int)
X = df.drop(columns=[TARGET_COL])


# Identify categorical and numerical columns
cat_cols = X.select_dtypes(exclude="number").columns.tolist()
num_cols = X.select_dtypes(include="number").columns.tolist()


# Encode categorical columns
encoders = {}

for c in cat_cols:
    le = LabelEncoder()
    X[c] = le.fit_transform(X[c].astype(str))
    encoders[c] = le


# Fill missing numerical values
imputer = SimpleImputer(strategy="median")
X[num_cols] = imputer.fit_transform(X[num_cols])


# Scale numerical columns
scaler = StandardScaler()
X[num_cols] = scaler.fit_transform(X[num_cols])


# Train / Validation / Test split
VAL_SIZE = 0.15
TEST_SIZE = 0.15


X_train_val, X_test, y_train_val, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE
)


val_ratio = VAL_SIZE / (1 - TEST_SIZE)


X_train, X_val, y_train, y_val = train_test_split(
    X_train_val,
    y_train_val,
    test_size=val_ratio,
    stratify=y_train_val,
    random_state=RANDOM_STATE
)


print("\nData split:")
print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)


# ==========================================
# BOOSTING BENCHMARK FUNCTION
# ==========================================

def boosting_benchmark(X_train, y_train, X_val, y_val):

    results = []


    # ======================================
    # AdaBoost
    # ======================================

    ada_base = DecisionTreeClassifier(
        max_depth=2,
        random_state=RANDOM_STATE
    )


    ada = AdaBoostClassifier(
        estimator=ada_base,
        n_estimators=200,
        learning_rate=0.5,
        random_state=RANDOM_STATE
    )


    t0 = time.time()

    ada.fit(X_train, y_train)

    ada_fit_time = time.time() - t0


    ada_val_pred = ada.predict(X_val)

    ada_val_proba = ada.predict_proba(X_val)[:, 1]


    results.append({
        "model": "AdaBoost",
        "val_accuracy": accuracy_score(
            y_val,
            ada_val_pred
        ),
        "val_f1": f1_score(
            y_val,
            ada_val_pred
        ),
        "val_roc_auc": roc_auc_score(
            y_val,
            ada_val_proba
        ),
        "best_n_estimators": ada.n_estimators,
        "fit_time_sec": round(
            ada_fit_time,
            2
        )
    })


    # ======================================
    # XGBoost
    # ======================================

    xgb = XGBClassifier(
        n_estimators=1000,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        early_stopping_rounds=30,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )


    t0 = time.time()


    xgb.fit(
        X_train,
        y_train,
        eval_set=[(X_val, y_val)],
        verbose=False
    )


    xgb_fit_time = time.time() - t0


    xgb_val_pred = xgb.predict(X_val)

    xgb_val_proba = xgb.predict_proba(X_val)[:, 1]


    results.append({
        "model": "XGBoost",
        "val_accuracy": accuracy_score(
            y_val,
            xgb_val_pred
        ),
        "val_f1": f1_score(
            y_val,
            xgb_val_pred
        ),
        "val_roc_auc": roc_auc_score(
            y_val,
            xgb_val_proba
        ),
        "best_n_estimators": xgb.best_iteration + 1,
        "fit_time_sec": round(
            xgb_fit_time,
            2
        )
    })


    # Create leaderboard
    results_df = pd.DataFrame(results).sort_values(
        "val_accuracy",
        ascending=False
    ).reset_index(drop=True)


    return results_df


# ==========================================
# RUN BENCHMARK
# ==========================================

leaderboard = boosting_benchmark(
    X_train,
    y_train,
    X_val,
    y_val
)


# ==========================================
# DISPLAY RESULTS
# ==========================================

print("\nValidation leaderboard:")
print(leaderboard.to_string(index=False))


# ==========================================
# SAVE RESULTS
# ==========================================

leaderboard.to_csv(
    "boosting_benchmark_results.csv",
    index=False
)

print("\nSaved results to boosting_benchmark_results.csv")

Dataset loaded successfully!
Dataset shape: (50000, 21)

Data split:
Train: (34999, 19)
Validation: (7501, 19)
Test: (7500, 19)

Validation leaderboard:
   model  val_accuracy   val_f1  val_roc_auc  best_n_estimators  fit_time_sec
AdaBoost      0.796294 0.780963     0.880162                200         11.28
 XGBoost      0.795894 0.782744     0.882638                135          0.89

Saved results to boosting_benchmark_results.csv
